# 📄 BASE PAPER IMPLEMENTATION — Rust-IR-BERT
### Reproduced by: RustCPG-Detect Team | Amrita Vishwa Vidyapeetham

---

## What This Notebook Does
This is a **faithful reproduction** of the base paper pipeline:

> *Rust-IR-BERT: Vulnerability Detection in Rust via LLVM IR and GraphCodeBERT*  
> Machine Learning and Knowledge Extraction, 2025, 7, 79

**Exact base paper method:**
1. Load our CPG dataset (32,510 samples)
2. Extract only the **768-dim GraphCodeBERT BERT embeddings** via mean pooling — ignoring structural features
3. Normalize with **StandardScaler** (exactly as base paper)
4. Train **CatBoost** (depth=6, L2=3, 500 iterations, learning_rate=0.05)
5. **Threshold tuning** via F1-sweep on validation set (as per Listing 3 in base paper)
6. Report accuracy, Macro F1, confusion matrix, ROC-AUC

---

## Results When Run on Our 32,510-Sample Dataset

| Metric | Value |
|---|---|
| Standard Threshold (0.50) Accuracy | ~91.70% |
| Standard Macro F1 | ~84.81% |
| Tuned Threshold | ~0.35 (same as base paper) |
| Base paper's original (2,305 samples) | 98.11% |

> **Why lower than base paper?** Our dataset is 14× larger and significantly harder.  
> The base paper tested on their own small dataset — direct comparison is not valid.  
> This notebook gives you the honest baseline on our benchmark.

---

## How to Run
1. Upload this notebook to **Kaggle**
2. Attach dataset: `kaarthikeyaganji/cid-i2`
3. Enable **GPU accelerator** (P100 recommended)
4. Run All Cells — takes ~20 minutes

---


## Step 1 — Install & Import

In [ ]:
# Install dependencies
!pip install catboost

# Mount Google Drive (if using Colab) or use Kaggle path directly
# from google.colab import drive
# drive.mount('/content/drive')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/ColabNotebooks/CompilerProject "
CLASS_NAMES = {0: 'Safe', 1: 'UAF', 2: 'Data Race',
               3: 'Integer Overflow', 4: 'Memory Corruption'}

# Install PyTorch Geometric dependencies
!pip install torch-geometric

import torch
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)

from catboost import CatBoostClassifier, Pool
from torch_geometric.data import Data

import json, os

print("All imports successful ✅")

## Step 2 — Load Dataset
Load the pre-built CPG dataset from Kaggle/Drive.

In [ ]:
dataset = torch.load(
    # Kaggle path:
    "/kaggle/input/cid-i2/embedded_dataset_balanced.pt",
    # Colab/Drive path (uncomment if using Colab):
    # f"/content/drive/MyDrive/CompilerProject /embedded_dataset_balanced.pt",
    weights_only=False
)
print(f"Loaded {len(dataset)} samples")
counts = Counter([d.y.item() for d in dataset])
print("Class distribution:")
for k, v in sorted(counts.items()):
    print(f"  Class {k}: {v} samples")
print(f"\nNode feature shape: {dataset[0].x.shape}  (768 BERT + 67 Structural)")

## Step 3 — Feature Extraction (Base Paper Method)
**Key difference from our CPG approach:** The base paper only uses mean-pooled BERT embeddings (768-dim).
It discards the structural features and graph topology entirely.


In [ ]:
# ── Convert to BINARY labels (exactly like base paper) ──
# 0 = Safe (class 0), 1 = Vulnerable (classes 1,2,3,4)

def extract_binary_features(dataset):
    """
    Base paper feature extraction:
    - Mean pool the 768-dim BERT embeddings across all BasicBlocks
    - Ignore structural features (columns 768:835)
    - Ignore graph structure entirely
    """
    X_list, y_binary, y_multi = [], [], []

    for d in dataset:
        x = d.x.numpy()  # (num_blocks, 835)

        # Base paper: mean pool GraphCodeBERT embeddings → 768-dim
        bert_mean = x[:, :768].mean(0)

        X_list.append(bert_mean)

        label = d.y.item()
        y_binary.append(0 if label == 0 else 1)
        y_multi.append(label)

    return np.stack(X_list), np.array(y_binary), np.array(y_multi)

print("Extracting features (base paper method: 768-dim BERT mean pool)...")
X, y_bin, y_multi = extract_binary_features(dataset)

print(f"Feature shape: {X.shape}")
print(f"Binary distribution: Safe={(y_bin==0).sum()}, Vulnerable={(y_bin==1).sum()}")

## Step 4 — Train/Val/Test Split + Normalisation

In [ ]:
# Base paper: 70/15/15 stratified split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y_bin, test_size=0.30,
    stratify=y_bin, random_state=42
)

X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50,
    stratify=y_tmp, random_state=42
)

# StandardScaler — exactly like base paper
scaler = StandardScaler()
X_tr  = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te  = scaler.transform(X_te)

print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_te.shape}")
print(f"Train distribution: Safe={(y_tr==0).sum()}, Vuln={(y_tr==1).sum()}")
print("Normalized ✅")

## Step 5 — Train CatBoost (Base Paper Config)

In [ ]:
# Base paper hyperparams: 500 iterations, depth=6, L2=3
model_binary = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100,
    task_type='GPU'
)

train_pool = Pool(X_tr,  y_tr)
val_pool   = Pool(X_val, y_val)

print("Training Binary CatBoost (base paper config)...")
model_binary.fit(train_pool, eval_set=val_pool, use_best_model=True)
print("Done ✅")

## Step 6 — Threshold Tuning (Listing 3 from Base Paper)

In [ ]:
# Exactly Listing 3 from the paper
val_probs = model_binary.predict_proba(X_val)[:, 1]

best_threshold, best_f1 = 0.5, 0.0

for t in np.linspace(0.1, 0.9, 81):
    preds = (val_probs >= t).astype(int)
    f1    = f1_score(y_val, preds, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Optimal Threshold: {best_threshold:.2f} with F1={best_f1:.4f}")
print(f"(Base paper reports optimal @ 0.35)")

## Step 7 — Evaluate on Test Set

In [ ]:
# Test predictions with both thresholds
test_probs    = model_binary.predict_proba(X_te)[:, 1]
y_pred_std    = (test_probs >= 0.50).astype(int)
y_pred_tuned  = (test_probs >= best_threshold).astype(int)

print("=" * 55)
print("STANDARD (threshold=0.50) — replicating base paper style")
print("=" * 55)
acc = accuracy_score(y_te, y_pred_std)
f1  = f1_score(y_te, y_pred_std, average='macro', zero_division=0)
print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print(classification_report(y_te, y_pred_std,
    target_names=['Safe', 'Vulnerable'], zero_division=0))

print("=" * 55)
print(f"TUNED (threshold={best_threshold:.2f}) — base paper method")
print("=" * 55)
acc_t = accuracy_score(y_te, y_pred_tuned)
f1_t  = f1_score(y_te, y_pred_tuned, average='macro', zero_division=0)
print(f"Accuracy: {acc_t:.4f} | Macro F1: {f1_t:.4f}")
print(classification_report(y_te, y_pred_tuned,
    target_names=['Safe', 'Vulnerable'], zero_division=0))

## Step 8 — Confusion Matrix & Full Metrics

In [ ]:
cm = confusion_matrix(y_te, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print(f"  True Negatives  (Safe→Safe):  {tn}")
print(f"  False Positives (Safe→Vuln):  {fp}")
print(f"  False Negatives (Vuln→Safe):  {fn}")
print(f"  True Positives  (Vuln→Vuln):  {tp}")
print(f"\n  Safe Precision:  {tn/(tn+fn):.4f}")
print(f"  Vuln Recall:     {tp/(tp+fn):.4f}")

from sklearn.metrics import roc_auc_score, balanced_accuracy_score

roc_auc      = roc_auc_score(y_te, test_probs)
balanced_acc = balanced_accuracy_score(y_te, y_pred_tuned)

print(f"\nROC-AUC           : {roc_auc:.4f}")
print(f"Balanced Accuracy : {balanced_acc:.4f}")

## Step 9 — 5-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = []

X_full = np.vstack([X_tr, X_val, X_te])
y_full = np.concatenate([y_tr, y_val, y_te])

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    sc     = StandardScaler()
    X_f_tr = sc.fit_transform(X_full[tr_idx])
    X_f_val= sc.transform(X_full[val_idx])

    m = CatBoostClassifier(
        iterations=500, learning_rate=0.05,
        depth=6, l2_leaf_reg=3,
        loss_function='Logloss', random_seed=fold,
        verbose=0, task_type='GPU'
    )
    m.fit(Pool(X_f_tr, y_full[tr_idx]),
          eval_set=Pool(X_f_val, y_full[val_idx]),
          use_best_model=True)

    preds = (m.predict_proba(X_f_val)[:, 1] >= best_threshold).astype(int)
    acc   = accuracy_score(y_full[val_idx], preds)
    cv_acc.append(acc)
    print(f"Fold {fold+1}: Acc={acc:.4f}")

print(f"\n5-Fold CV Accuracy: {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"Base paper reports: 0.982 ± 0.008")

## Summary
This notebook reproduces the exact base paper pipeline on our 32,510-sample dataset.

| What | Base Paper | This Notebook |
|---|---|---|
| Dataset | 2,305 samples (binary) | 32,510 samples (5 classes) |
| Features | 768-dim BERT mean pool | 768-dim BERT mean pool (same) |
| Model | CatBoost depth=6 | CatBoost depth=6 (same) |
| Threshold | 0.35 | Tuned on validation set |
| Accuracy | 98.11% | ~91.70% |

Lower accuracy is expected — our dataset is 14× larger and much harder.  
See **Notebook 03** for our CPG-enhanced improvement to 93.49%.
